<a href="https://colab.research.google.com/github/M1tayka/PIRSMA_M/blob/dz4/dz4m.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
pip install pyspark numpy

In [38]:
from google.colab import files
uploaded = files.upload()

Saving links.csv to links (2).csv
Saving movies.csv to movies (2).csv
Saving ratings.csv to ratings (2).csv
Saving tags.csv to tags (2).csv


In [39]:
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("MovieLens_ALS_Factorization") \
    .master("local[*]") \
    .getOrCreate()

In [40]:

ratings_df = spark.read.csv("ratings.csv", header=True, inferSchema=True)
movies_df = spark.read.csv("movies.csv", header=True, inferSchema=True)

ratings = ratings_df.select("userId", "movieId", "rating").cache()

print(f"всего оценок- {ratings.count()}")

всего оценок- 100836


In [41]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True
)

In [42]:
param_grid = ParamGridBuilder() \
    .addGrid(als.rank, [5, 10, 15]) \
    .addGrid(als.regParam, [0.001, 0.01, 0.1, 1, 10]) \
    .build()

In [43]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

In [44]:
cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=4
)

In [46]:
cv_model = cv.fit(ratings)

In [47]:
best_model = cv_model.bestModel

print("\nлучшие параметры модели")
print(f"количество факторов - {best_model.rank}")
print(f"регуляризация - {best_model._java_obj.parent().getRegParam()}")


лучшие параметры модели
количество факторов - 15
регуляризация - 0.1


In [48]:
predictions = best_model.transform(ratings)
rmse = evaluator.evaluate(predictions)
print(f"rsme лучшей модели на всех данных: {rmse:.4f}")

print("\n сравнение предсказанных и реальных рейтингов ")
predictions.join(movies_df, "movieId") \
    .select("userId", "title", "rating", "prediction") \
    .show(10, truncate=False)

print("\nгенерация персональных рекомендаций")

user_recs = best_model.recommendForAllUsers(5)

user_1_recs = user_recs.filter(col("userId") == 1).select("recommendations").collect()[0][0]

print(f"топ 5 рекомендаций для пользователя 1:")

rec_movie_ids = [row.movieId for row in user_1_recs]
movies_df.filter(col("movieId").isin(rec_movie_ids)).select("movieId", "title").show(truncate=False)

spark.stop()

rsme лучшей модели на всех данных: 0.5845

 сравнение предсказанных и реальных рейтингов 
+------+----------------------------------------------------------------------------------------------+------+----------+
|userId|title                                                                                         |rating|prediction|
+------+----------------------------------------------------------------------------------------------+------+----------+
|148   |Forrest Gump (1994)                                                                           |4.0   |3.5174227 |
|148   |Princess Bride, The (1987)                                                                    |3.0   |3.5687895 |
|148   |Moulin Rouge (2001)                                                                           |4.0   |3.5819883 |
|148   |Monsters, Inc. (2001)                                                                         |3.0   |3.565363  |
|148   |Harry Potter and the Sorcerer's Stone (a.k.a. Ha